# SFT with Synthetic Data and Tool Use

Convert an OpenAI tool catalog to OpenAPI 3.0, generate tool-use examples with Foundry's
`ToolUseFineTuning` recipe, validate structural correctness, fine-tune a student, and compare it
with the base model.

**Critical lineage rule:** the notebook always submits live data generation. Validation, splitting,
evaluation, and training consume the downloaded output from those exact jobs. The preserved OpenAPI
file is a reference fixture, not a generated-data fallback.

## 1. Setup and source schemas

The OpenAI-format catalog and its preserved OpenAPI conversion are byte-for-byte migrated and
hashed. The notebook performs a fresh conversion, compares operation IDs with the preserved
reference, uploads the fresh result, then generates training examples.

In [ ]:
import json
import os
import random
import sys
from datetime import datetime, timezone
from pathlib import Path

from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv

load_dotenv()
ROOT = Path.cwd()
DATA = ROOT / "data"
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
ARTIFACTS = ROOT / "outputs" / RUN_ID
ARTIFACTS.mkdir(parents=True, exist_ok=False)
TRACK_ROOT = next((path for path in (ROOT, *ROOT.parents) if (path / "stage05_runtime.py").is_file()), None)
if TRACK_ROOT is None:
    raise RuntimeError("Run this notebook from its demo directory inside 05-data-and-distillation.")
sys.path.insert(0, str(TRACK_ROOT))
from stage05_runtime import monitor_fine_tuning_job, upload_file_and_wait

PROJECT_ENDPOINT = os.environ["FOUNDRY_PROJECT_ENDPOINT"]
FOUNDRY_RUN_PAID_JOBS = os.environ.get("FOUNDRY_RUN_PAID_JOBS", "false").lower() == "true"
FILE_TIMEOUT_SECONDS = float(os.environ.get("FOUNDRY_FILE_TIMEOUT_SECONDS", "600"))
JOB_TIMEOUT_SECONDS = float(os.environ.get("FOUNDRY_JOB_TIMEOUT_SECONDS", "86400"))
POLL_INTERVAL_SECONDS = float(os.environ.get("FOUNDRY_POLL_INTERVAL_SECONDS", "10"))

credential = DefaultAzureCredential()
project = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential, allow_preview=True)
model_client = project.get_openai_client()

def read_jsonl(path):
    with Path(path).open(encoding="utf-8") as handle:
        return [json.loads(line) for line in handle if line.strip()]

def write_jsonl(path, rows):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8", newline="\n") as handle:
        for row in rows:
            handle.write(json.dumps(row, ensure_ascii=False) + "\n")
    return path

print("Configuration loaded. Paid training enabled:", FOUNDRY_RUN_PAID_JOBS)

TEACHER_DEPLOYMENT = os.environ["FOUNDRY_TEACHER_DEPLOYMENT"]
STUDENT_DEPLOYMENT = os.environ["FOUNDRY_STUDENT_DEPLOYMENT"]
STUDENT_MODEL = os.environ.get("FOUNDRY_STUDENT_MODEL", "gpt-4.1-mini-2025-04-14")
FINE_TUNED_DEPLOYMENT = os.environ.get("FOUNDRY_FINE_TUNED_DEPLOYMENT", "")
FOUNDRY_MAX_SAMPLES = int(os.environ.get("FOUNDRY_MAX_SAMPLES", "60"))
FOUNDRY_GENERATION_BATCHES = int(os.environ.get("FOUNDRY_GENERATION_BATCHES", "3"))
FOUNDRY_TRAINING_TYPE = os.environ.get("FOUNDRY_TRAINING_TYPE", "globalStandard")
FOUNDRY_RUN_LIVE_GENERATION = os.environ.get("FOUNDRY_RUN_LIVE_GENERATION", "false").lower() == "true"
FOUNDRY_RUN_LIVE_EVALUATION = os.environ.get("FOUNDRY_RUN_LIVE_EVALUATION", "false").lower() == "true"

OPENAI_TOOLS = json.loads((DATA / "zava_tools_openai.json").read_text(encoding="utf-8"))
OPENAPI_REFERENCE = json.loads((DATA / "zava_tools_openapi.json").read_text(encoding="utf-8"))
assert OPENAI_TOOLS and OPENAPI_REFERENCE["openapi"].startswith("3.")
print("Tool count:", len(OPENAI_TOOLS))

## 2. OpenAI tool schema to OpenAPI conversion

In [ ]:
def openai_tools_to_openapi(tools, title="Tool catalog"):
    paths = {}
    for tool in tools:
        if tool.get("type") != "function":
            raise ValueError(f"Unsupported tool type: {tool.get('type')}")
        function = tool["function"]
        name = function["name"]
        paths[f"/{name}"] = {"post": {
            "operationId": name,
            "summary": function.get("description", name)[:120],
            "description": function.get("description", ""),
            "requestBody": {
                "required": True,
                "content": {"application/json": {"schema": function.get("parameters", {"type": "object"})}},
            },
            "responses": {"200": {
                "description": "Tool response",
                "content": {"application/json": {"schema": {"type": "object"}}},
            }},
        }}
    return {
        "openapi": "3.0.3",
        "info": {"title": title, "version": "1.0.0"},
        "servers": [{"url": "https://example.invalid"}],
        "paths": paths,
    }

openapi_spec = openai_tools_to_openapi(OPENAI_TOOLS, "Zava tool catalog")
reference_ops = {item["post"]["operationId"] for item in OPENAPI_REFERENCE["paths"].values()}
generated_ops = {item["post"]["operationId"] for item in openapi_spec["paths"].values()}
assert generated_ops == reference_ops
OPENAPI_PATH = ARTIFACTS / "zava_tools_openapi.json"
OPENAPI_PATH.write_text(json.dumps(openapi_spec, indent=2), encoding="utf-8")
print("Converted operations:", len(generated_ops))

## 3. Generate exact training candidates

The Foundry data-generation service uses the teacher and uploaded OpenAPI spec. Job IDs and recipe
parameters are recorded beside every downloaded batch. If generation fails, the notebook stops; it
does not train on prior run files.

In [ ]:
def submit_tool_use_datagen(spec_file_id, output_path):
    from azure.ai.projects.models import (
        DataGenerationJob,
        DataGenerationJobInputs,
        DataGenerationJobOutputOptions,
        DataGenerationJobScenario,
        DataGenerationModelOptions,
        FileDataGenerationJobSource,
        ToolUseFineTuningDataGenerationJobOptions,
    )
    options = ToolUseFineTuningDataGenerationJobOptions(
        model_options=DataGenerationModelOptions(model=TEACHER_DEPLOYMENT),
        max_samples=FOUNDRY_MAX_SAMPLES,
    )
    output_name = Path(output_path).stem
    job = DataGenerationJob(inputs=DataGenerationJobInputs(
        name=output_name,
        scenario=DataGenerationJobScenario.SUPERVISED_FINETUNING,
        sources=[FileDataGenerationJobSource(id=spec_file_id)],
        options=options,
        output_options=DataGenerationJobOutputOptions(name=output_name),
    ))
    poller = project.beta.datasets.begin_create_generation_job(job=job)
    job_id = poller.details.get("job_id")
    if not job_id:
        raise RuntimeError("Data-generation submission did not return a job ID.")
    result = poller.result()
    outputs = result.outputs or []
    if not outputs:
        raise RuntimeError(f"Data-generation job {job_id} completed without output files.")
    with Path(output_path).open("wb") as destination:
        for output in outputs:
            destination.write(model_client.files.content(file_id=output.id).read())
    metadata = {
        "job_id": job_id,
        "output_file_ids": [output.id for output in outputs],
        "generated_samples": result.generated_samples,
        "teacher_deployment": TEACHER_DEPLOYMENT,
        "recipe": "ToolUseFineTuning",
        "max_samples": FOUNDRY_MAX_SAMPLES,
        "generated_at": datetime.now(timezone.utc).isoformat(),
    }
    Path(str(output_path) + ".provenance.json").write_text(
        json.dumps(metadata, indent=2), encoding="utf-8"
    )
    return Path(output_path)

if not FOUNDRY_RUN_LIVE_GENERATION:
    raise RuntimeError("Set FOUNDRY_RUN_LIVE_GENERATION=true to submit live data generation. No snapshot fallback is used.")
spec_file = upload_file_and_wait(
    model_client, OPENAPI_PATH, purpose="user_data",
    poll_interval_seconds=POLL_INTERVAL_SECONDS, timeout_seconds=FILE_TIMEOUT_SECONDS,
)

GENERATED_PATHS = [
    submit_tool_use_datagen(
        spec_file.id, ARTIFACTS / f"tool_use_generated_{batch}.jsonl"
    )
    for batch in range(1, FOUNDRY_GENERATION_BATCHES + 1)
]
generated_rows_by_content = {}
for generated_path in GENERATED_PATHS:
    for row in read_jsonl(generated_path):
        generated_rows_by_content.setdefault(json.dumps(row, sort_keys=True), row)
generated_rows = list(generated_rows_by_content.values())
if not generated_rows:
    raise RuntimeError("The live data-generation job produced no rows.")
print("Downloaded live generated rows:", len(generated_rows))

## 4. Validate, filter, and split the downloaded output

In [ ]:
def validate_tool_row(row):
    messages = row.get("messages")
    if not isinstance(messages, list) or len(messages) < 2:
        raise ValueError("Each row must contain a messages array.")
    assistant_calls = [
        call
        for message in messages
        if message.get("role") == "assistant"
        for call in message.get("tool_calls", [])
    ]
    if not assistant_calls:
        raise ValueError("Each accepted row must teach at least one assistant tool call.")
    for call in assistant_calls:
        function = call.get("function", {})
        if function.get("name") not in generated_ops:
            raise ValueError(f"Unknown tool: {function.get('name')}")
        json.loads(function.get("arguments", "{}"))
    return True

accepted = [row for row in generated_rows if validate_tool_row(row)]
random.Random(42).shuffle(accepted)
if len(accepted) < 20:
    raise RuntimeError("Fewer than 20 unique valid generated rows; inspect the generation jobs.")
n_train, n_validation = int(0.8 * len(accepted)), max(1, int(0.1 * len(accepted)))
train_rows = accepted[:n_train]
validation_rows = accepted[n_train:n_train + n_validation]
test_rows = accepted[n_train + n_validation:]
TRAIN_PATH = write_jsonl(ARTIFACTS / "train.jsonl", train_rows)
VALIDATION_PATH = write_jsonl(ARTIFACTS / "validation.jsonl", validation_rows)
TEST_PATH = write_jsonl(ARTIFACTS / "test.jsonl", test_rows)
print(len(train_rows), len(validation_rows), len(test_rows))

## 5. Structural evaluation

Compare tool name, parsed arguments, and final outcome. Exact argument matches score highest;
partial name overlap is diagnostic but not a pass. Reuse the same held-out rows before and after
fine-tuning.

In [ ]:
def normalized_calls(tool_calls):
    calls = []
    for call in tool_calls:
        function = call.get("function", {}) if isinstance(call, dict) else call.function
        name = function.get("name") if isinstance(function, dict) else function.name
        arguments = function.get("arguments", "{}") if isinstance(function, dict) else function.arguments
        calls.append((name, json.loads(arguments or "{}") if isinstance(arguments, str) else arguments))
    return calls

def tool_call_score(expected, actual):
    if expected == actual:
        return 10
    expected_names = [name for name, _ in expected]
    actual_names = [name for name, _ in actual]
    if expected_names == actual_names:
        return 8
    overlap = len(set(expected_names) & set(actual_names))
    return 1 if not overlap else round(2 + 6 * overlap / len(set(expected_names)), 2)

def evaluation_turns(rows):
    turns = []
    for row in rows:
        for index, message in enumerate(row["messages"]):
            calls = message.get("tool_calls") if message.get("role") == "assistant" else None
            if calls:
                turns.append({"messages": row["messages"][:index], "expected": normalized_calls(calls)})
    if not turns:
        raise RuntimeError("The held-out split contains no assistant tool-call turns.")
    return turns

def predict_calls(deployment, messages):
    response = model_client.chat.completions.create(
        model=deployment, messages=messages, tools=OPENAI_TOOLS,
        temperature=0, max_completion_tokens=2048,
    )
    return normalized_calls(response.choices[0].message.tool_calls or [])

def evaluate_tool_deployment(deployment, turns):
    scores = []
    for turn in turns:
        scores.append(tool_call_score(turn["expected"], predict_calls(deployment, turn["messages"])))
    return {"count": len(scores), "average": sum(scores) / len(scores), "scores": scores}

TEST_TURNS = evaluation_turns(test_rows)
print("Structural evaluator ready for", len(TEST_TURNS), "held-out tool-call turns")

## 6. Fine-tune, deployment validation, and comparison

In [ ]:
if not FOUNDRY_RUN_PAID_JOBS:
    print("Training skipped. The generated files remain available for review.")
else:
    train_file = upload_file_and_wait(
        model_client, TRAIN_PATH, purpose="fine-tune",
        poll_interval_seconds=POLL_INTERVAL_SECONDS, timeout_seconds=FILE_TIMEOUT_SECONDS,
    )
    validation_file = upload_file_and_wait(
        model_client, VALIDATION_PATH, purpose="fine-tune",
        poll_interval_seconds=POLL_INTERVAL_SECONDS, timeout_seconds=FILE_TIMEOUT_SECONDS,
    )
    job = model_client.fine_tuning.jobs.create(
        model=STUDENT_MODEL,
        training_file=train_file.id,
        validation_file=validation_file.id,
        hyperparameters={"n_epochs": 3, "learning_rate_multiplier": 1.0},
        suffix="synthetic-tool-use",
        extra_body={"trainingType": FOUNDRY_TRAINING_TYPE},
    )
    print("Fine-tuning submission accepted; monitoring training to a terminal state.")
    completed_job = monitor_fine_tuning_job(
        model_client, job.id,
        poll_interval_seconds=POLL_INTERVAL_SECONDS, timeout_seconds=JOB_TIMEOUT_SECONDS,
    )
    print("Training outcome:", completed_job.status)

if FINE_TUNED_DEPLOYMENT:
    deployment = project.deployments.get(FINE_TUNED_DEPLOYMENT)
    print("Fine-tuned deployment found:", deployment.name)
else:
    print("Create a project deployment for the completed model, then set FOUNDRY_FINE_TUNED_DEPLOYMENT.")

if FOUNDRY_RUN_LIVE_EVALUATION:
    comparison = {"student_base": evaluate_tool_deployment(STUDENT_DEPLOYMENT, TEST_TURNS)}
    if FINE_TUNED_DEPLOYMENT:
        comparison["student_fine_tuned"] = evaluate_tool_deployment(FINE_TUNED_DEPLOYMENT, TEST_TURNS)
    (ARTIFACTS / "evaluation.json").write_text(json.dumps(comparison, indent=2), encoding="utf-8")
    print({name: result["average"] for name, result in comparison.items()})
else:
    print("Set FOUNDRY_RUN_LIVE_EVALUATION=true to run the same structural test before and after fine-tuning.")

## 7. Representative existing-run gains

On the preserved 15-tool retail catalog, the recorded run improved the combined structural score
from **7.00** to **9.00**, pass rate from **66.7%** to **88.9%**, and reported **+28.6% lift**.

## Conclusion and next step

A tool catalog can bootstrap SFT data, but tool-name accuracy alone is insufficient: validate
arguments, multi-call ordering, confirmations, and final user-visible outcomes. Data-generation
completion, fine-tuning submission, successful monitored training, deployment, and held-out
evaluation are distinct outcomes. Next, add domain-specific negative cases and policy-sensitive
confirmation flows.

*Variability footnote: data generation, service recipes, model versions, split composition, and
tool schemas can change measured lift. Version the OpenAPI document, generated output, filters,
job IDs, and evaluation rows together.*

> **Variability note:** Actual results may vary by model version, data, configuration, region availability, and service conditions.